### Part 1

In [1]:
!pytest tests/test_loss.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_loss.py ...                                                   [100%]

============================== 3 passed in 3.30s ===============================


In [2]:
!pytest tests/test_attention.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 4 items                                                              

tests/test_attention.py ....                                             [100%]

=============================== warnings summary ===============================
tests/test_attention.py::TestAttentionCorrectness::test_backward[dtype0-0.05-0.05]
  /workspace/efdl_hw/week06_dl_arithmetic/homework/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

-- Docs: https://docs.pyte

In [3]:
!pytest tests/test_rmsnorm.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_rmsnorm.py ...                                                [100%]

============================== 3 passed in 3.12s ===============================


In [4]:
!pytest tests/test_swiglu.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 4 items                                                              

tests/test_swiglu.py ....                                                [100%]

=============================== warnings summary ===============================
tests/test_swiglu.py::TestSwiGLUCorrectness::test_backward[dtype0-5e-05-5e-05]
  /workspace/efdl_hw/week06_dl_arithmetic/homework/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

-- Docs: https://docs.pytest.o

In [5]:
!pytest tests/test_e2e.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_e2e.py ...                                                    [100%]

============================== 3 passed in 6.20s ===============================


### Part 2

### 1-2

In [8]:
!pytest tests/test_optimizer.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 1 item                                                               

tests/test_optimizer.py .                                                [100%]

============================== 1 passed in 2.93s ===============================


In [9]:
!cat efficient_optimizer/ademamix.py

import math
 
import torch
from torch import Tensor
from torch.distributed.tensor import DTensor
from torch.optim import Optimizer
from typing import Optional
 
# HINT: you may want to change these functions
def linear_warmup_scheduler(step, alpha_end, alpha_start=0, warmup=1):
    return torch.where(
        step < warmup,
        (1.0-step / float(warmup)) * alpha_start + step / float(warmup) * alpha_end,
        alpha_end
    )


def linear_hl_warmup_scheduler(step, beta_end, beta_start=0, warmup=1):
    def f(beta, eps=1e-8):
        return math.log(0.5)/torch.log(beta+eps)-1

    return torch.where(
        step < warmup,
        torch.pow(0.5, 1/(((1.0-step / float(warmup)) * f(beta_start) + step / float(warmup) * f(beta_end))+1)),
        beta_end
    )


@torch.compile(fullgraph=True) # you can comment out this line for subtask 1
def ademamix_foreach_fn(
    params: list[Tensor],
    grads: list[Tensor],
    exp_avg_fasts: list[Tensor],
    exp_avg_slows: list[Tensor],
    exp_

In [1]:
import os
os.environ["TORCH_LOGS"] = "+output_code"
os.environ["TORCH_LOGS_OUT"] = "compiled_kernels.log"

In [2]:
import pytest
import torch
import torch.nn as nn

from optimizer.ademamix import AdEMAMix as AdemamixForloop
from efficient_optimizer.ademamix import AdEMAMix as AdemamixForeach

import torch._dynamo as dynamo
dynamo.config.recompile_limit = 8

HIDDEN_DIM = 16
NUM_LAYERS = 3
NUM_STEPS = 100

def _build_model(device: torch.device, dtype: torch.dtype) -> nn.Module:
    torch.manual_seed(0)
    layers = [nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=True) for _ in range(NUM_LAYERS)]
    return nn.Sequential(*layers).to(device=device, dtype=dtype)


def _assert_models_close(model_a: nn.Module, model_b: nn.Module, step: int, rtol: float=1e-5, atol: float=1e-6) -> None:
    a = dict(model_a.named_parameters())
    b = dict(model_b.named_parameters())
    assert a.keys() == b.keys()

    for name in a.keys():
        pa, pb = a[name].data, b[name].data
        max_diff = (pa - pb).abs().max().item()
        assert torch.allclose(pa, pb, atol=atol, rtol=rtol), (
            f"Param mismatch at step={step}, name={name}, max_diff={max_diff}"
        )

In [3]:
def _apply_random_grads(model_a: nn.Module, model_b: nn.Module) -> None:
    torch.manual_seed(0)
    a = dict(model_a.named_parameters())
    b = dict(model_b.named_parameters())
    for name in a.keys():
        g = torch.randn_like(a[name].data)
        a[name].grad = g
        b[name].grad = g.clone()

In [4]:
device = 'cuda'
dtype = torch.float32

model_efficient = _build_model(device, dtype)

opt_efficient = AdemamixForeach(model_efficient.parameters(), lr=1e-2, weight_decay=0.1, alpha_warmup=51, beta3_warmup=51)

In [5]:
model_with_adam = _build_model(device, dtype)
opt_adam = torch.optim.Adam(model_with_adam.parameters(), lr=1e-2, weight_decay=0.1, foreach=True)

In [6]:
torch._dynamo.reset()
_apply_random_grads(model_efficient, model_with_adam)
opt_efficient.step()

V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] Output code: 
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] # AOT ID: ['1_inference']
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import torch
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import math
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import random
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import os
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import tempfile
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236] [0/0] [__output_code] from math import inf, nan
V0312 18:09:17.903000 94939 torch/_inductor/codecache.py:1236

In [7]:
torch._dynamo.reset()
opt_adamw = torch.optim.AdamW(model_with_adam.parameters(), lr=1e-2, weight_decay=0.1, foreach=True)
compiled_adamw_step = torch.compile(opt_adamw.step)
_apply_random_grads(model_efficient, model_with_adam)
compiled_adamw_step()

W0312 18:09:17.971000 94939 torch/_logging/_internal.py:1154] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] Output code: 
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] # AOT ID: ['1_inference']
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import torch
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import math
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import random
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import os
V0312 18:09:18.208000 94939 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import tempfile
V0312 18:09:18.20800

**у адама там отдельно считается керел для скаляров (типо шага) и для тензоров**

у меня щас 3 кернела, но 2 из них для скаляров, так что чтобы получить 1 поможет вытащить все скалярные операции из кернела